In [ ]:
import os
import json
import time
import requests
import re
import sqlite3
from dotenv import load_dotenv

# 1. 환경 설정
load_dotenv(override=True)
UPSTAGE_API_KEY = os.environ.get("UPSTAGE_API_KEY")
DB_PATH = 'database/ESGdata.db'
IMAGE_DIR = './ocr_results_folder'            # 고지서 이미지들이 들어있는 폴더
OCR_RESULT_DIR = './ocr_results' # JSON 결과가 저장될 폴더

def run_upstage_ocr_batch(image_path):
    """Upstage API를 호출하여 OCR 결과를 반환합니다."""
    url = "https://api.upstage.ai/v1/document-digitization"
    headers = {"Authorization": f"Bearer {UPSTAGE_API_KEY}"}
    
    try:
        with open(image_path, "rb") as f:
            files = {"document": f}
            data = {"model": "ocr"}
            response = requests.post(url, headers=headers, files=files, data=data)
            
        if response.status_code == 200:
            return response.json()
        else:
            print(f"❌ API 호출 실패 ({os.path.basename(image_path)}): {response.status_code}")
            return None
    except Exception as e:
        print(f"❌ 예외 발생 ({os.path.basename(image_path)}): {e}")
        return None

def parse_ocr_logic(ocr_result):
    """팀장님의 정규표현식 파싱 로직"""
    full_text = ocr_result.get('content', {}).get('text', '') or ocr_result.get('text', '')
    
    cust_no = re.search(r'고객번호\s*([\d\s]{5,20})', full_text)
    date_info = re.search(r'(\d{4})\s*년\s*(\d{1,2})\s*월분', full_text)
    usage_info = re.search(r'(?:검침량|당\s*월)\s*([\d,.]+)\s*(kWh|m³|Nm3|Wh|L|MWh)?', full_text)
    
    return {
        "customer_number": cust_no.group(1).replace(" ", "").strip() if cust_no else None,
        "year": date_info.group(1) if date_info else None,
        "month": f"{int(date_info.group(2)):02d}" if date_info else None,
        "usage": float(usage_info.group(1).replace(",", "")) if usage_info else 0.0,
        "unit": usage_info.group(2) if usage_info and usage_info.group(2) else "unknown",
        "raw_json": ocr_result
    }

def main_process():
    # 저장 폴더 생성
    os.makedirs(OCR_RESULT_DIR, exist_ok=True)

    # 1. 폴더 내 이미지 파일 목록 가져오기
    valid_extensions = ('.jpg', '.jpeg', '.png', '.pdf')
    image_files = [f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(valid_extensions)]
    
    if not image_files:
        print(f"💡 {IMAGE_DIR} 폴더에 처리할 이미지 파일이 없습니다.")
        return

    print(f"📂 총 {len(image_files)}개의 파일을 발견했습니다. 분석을 시작합니다.")

    with sqlite3.connect(DB_PATH) as conn:
        for idx, file_name in enumerate(image_files, 1):
            image_path = os.path.join(IMAGE_DIR, file_name)
            
            # 🚀 1. 요청 전후로 1~2초간 대기 (429 에러 방지 핵심)
            if idx > 1:
                print("⏳ API 안정성을 위해 잠시 대기합니다 (1.5초)...")
                time.sleep(1.5) 

            print(f"🚀 [{idx}/{len(image_files)}] {file_name} 분석 중...")
            ocr_result = run_upstage_ocr_batch(image_path)
            
            if ocr_result:
                # 3. 개별 JSON 파일 저장 (파일명 유지)
                json_filename = os.path.splitext(file_name)[0] + ".json"
                json_path = os.path.join(OCR_RESULT_DIR, json_filename)
                with open(json_path, "w", encoding="utf-8") as f:
                    json.dump(ocr_result, f, ensure_ascii=False, indent=4)



                # 4. 정규표현식 파싱
                parsed = parse_ocr_logic(ocr_result)
                
                # 5. DB 적재 (raw_ocr_data 테이블)
                conn.execute("""
                    INSERT INTO raw_ocr_data (file_name, raw_content, ocr_provider, processing_status)
                    VALUES (?, ?, ?, ?)
                """, (
                    file_name, 
                    json.dumps(parsed, ensure_ascii=False), 
                    'Upstage_OCR', 
                    'Pending'
                ))
                print(f"✅ {file_name} 처리 및 DB 적재 완료")
            else:
                print(f"❌ {file_name} 처리 실패하여 다음 파일로 넘어갑니다.")
        
        conn.commit()
    print("\n🏁 모든 파일의 수집 및 DB 이관이 완료되었습니다.")

if __name__ == "__main__":
    main_process()

📂 총 2개의 파일을 발견했습니다. 분석을 시작합니다.
🚀 [1/2] 가스사용 고지서.jpg 분석 중...
✅ 가스사용 고지서.jpg 처리 및 DB 적재 완료
⏳ API 안정성을 위해 잠시 대기합니다 (1.5초)...
🚀 [2/2] 한전 고지서.jpg 분석 중...
✅ 한전 고지서.jpg 처리 및 DB 적재 완료

🏁 모든 파일의 수집 및 DB 이관이 완료되었습니다.


In [ ]:
DB_PATH = 'database/ESGdata.db'

def run_integrated_verification():
    with sqlite3.connect(DB_PATH) as conn:
        cursor = conn.cursor()
        
        # 1. 아직 처리되지 않은 Pre-parsed OCR 데이터 가져오기
        pending_data = cursor.execute("""
            SELECT id, file_name, raw_content 
            FROM raw_ocr_data 
            WHERE processing_status = 'Pending'
        """).fetchall()
        
        print(f"🚀 총 {len(pending_data)}건의 증빙 이관 및 검증을 시작합니다.")

        for r_id, f_name, raw_content in pending_data:
            parsed = json.loads(raw_content)
            
            # [Step A] evidence_usage 테이블에 정제된 데이터 적재 (누락된 부분 추가)
            cursor.execute("""
                INSERT INTO evidence_usage (site_id, reporting_date, metric, ocr_value, ocr_unit, file_name)
                SELECT map.site_id, ?, map.metric, ?, ?, ?
                FROM site_metric_map map
                WHERE map.customer_number = ?
            """, (
                f"{parsed['year']}-{parsed['month']}", 
                parsed['usage'], 
                parsed['unit'], 
                f_name, 
                parsed['customer_number']
            ))
            
            evidence_id = cursor.lastrowid # 방금 생성된 증빙 ID

            # [Step B] 팀장님의 reconcile_data 로직 수행 (실적 조회)
            reporting_date = f"{parsed['year']}-{parsed['month']}"
            query_std = """
                SELECT s.id, s.value, map.site_id, map.metric
                FROM standard_usage s
                JOIN site_metric_map map ON s.site_id = map.site_id AND s.metric = map.metric
                WHERE map.customer_number = ? AND s.reporting_date = ?
            """
            cursor.execute(query_std, (parsed['customer_number'], reporting_date))
            std_record = cursor.fetchone()

            if std_record:
                std_id, db_val, site_id, metric = std_record
                ocr_val = parsed['usage']
                gap = db_val - ocr_val
                abs_gap = abs(gap)

                # [Step C] 정합성 판정 및 v_status 결정
                if abs_gap < 1.0:
                    final_status = 5
                    diag = "✅ 정합성 확인 완료"
                elif abs(db_val * 1000 - ocr_val) < 1.0 or abs(db_val / 1000 - ocr_val) < 1.0:
                    final_status = 4
                    diag = "⚠️ 단위 기입 오류"
                else:
                    final_status = 3
                    diag = f"❌ 수치 불일치 (차이: {gap:.2f})"

                # standard_usage 및 verification_logs 업데이트
                cursor.execute("UPDATE standard_usage SET v_status = ? WHERE id = ?", (final_status, std_id))
                cursor.execute("""
                    INSERT INTO verification_logs (std_id, evidence_id, gap_value, result_code, diagnosis)
                    VALUES (?, ?, ?, ?, ?)
                """, (std_id, evidence_id, gap, final_status, diag))
                
                print(f"[{site_id}] {reporting_date} {metric} -> {diag}")

            # 처리 완료 표시
            cursor.execute("UPDATE raw_ocr_data SET processing_status = 'Success' WHERE id = ?", (r_id,))
        
        conn.commit()
    print("🏁 모든 증빙 이관 및 검증 프로세스가 완료되었습니다.")

if __name__ == "__main__":
    run_integrated_verification()

🚀 총 45건의 증빙 이관 및 검증을 시작합니다.
[Site A] 2025-01 electricity_mwh -> ✅ 정합성 확인 완료
[Site A] 2025-01 lng_nm3 -> ✅ 정합성 확인 완료
[Site A] 2025-02 electricity_mwh -> ✅ 정합성 확인 완료
[Site A] 2025-02 lng_nm3 -> ✅ 정합성 확인 완료
[Site A] 2025-03 electricity_mwh -> ❌ 수치 불일치 (차이: -150.00)
[Site A] 2025-03 lng_nm3 -> ✅ 정합성 확인 완료
[Site A] 2025-04 electricity_mwh -> ✅ 정합성 확인 완료
[Site A] 2025-04 lng_nm3 -> ✅ 정합성 확인 완료
[Site A] 2025-05 electricity_mwh -> ✅ 정합성 확인 완료
[Site A] 2025-05 lng_nm3 -> ✅ 정합성 확인 완료
[Site A] 2025-06 electricity_mwh -> ✅ 정합성 확인 완료
[Site A] 2025-06 lng_nm3 -> ✅ 정합성 확인 완료
[Site A] 2025-07 lng_nm3 -> ✅ 정합성 확인 완료
[Site A] 2025-08 electricity_mwh -> ✅ 정합성 확인 완료
[Site A] 2025-08 lng_nm3 -> ✅ 정합성 확인 완료
[Site A] 2025-09 electricity_mwh -> ✅ 정합성 확인 완료
[Site A] 2025-09 lng_nm3 -> ✅ 정합성 확인 완료
[Site A] 2025-10 lng_nm3 -> ✅ 정합성 확인 완료
[Site A] 2025-11 electricity_mwh -> ✅ 정합성 확인 완료
[Site A] 2025-11 lng_nm3 -> ✅ 정합성 확인 완료
[Site A] 2025-12 electricity_mwh -> ✅ 정합성 확인 완료
[Site A] 2025-12 lng_nm3 -> ✅ 정합성 확인 완료
[

In [ ]:
# import sqlite3
# import json
# import pandas as pd

# DB_PATH = 'database/ESGdata.db'

# def inject_preparsed_mock_data_final():
#     with sqlite3.connect(DB_PATH) as conn:
#         cursor = conn.cursor()
        
#         # 1. 기존 데이터 초기화 (깔끔한 테스트 환경 구축)
#         cursor.execute("DELETE FROM raw_ocr_data")
        
#         # 2. 검증 대상(2025년 v_status=1) 가져오기
#         query = """
#             SELECT s.id, s.site_id, s.metric, s.reporting_date, s.value, m.customer_number
#             FROM standard_usage s
#             JOIN site_metric_map m ON s.site_id = m.site_id AND s.metric = m.metric
#             WHERE s.reporting_date LIKE '2025-%' AND s.v_status = 1
#         """
#         targets = pd.read_sql(query, conn)
        
#         mock_records = []
#         for _, row in targets.iterrows():
#             site, metric, date, val, cust_no = row['site_id'], row['metric'], row['reporting_date'], row['value'], row['customer_number']
#             year, month = date.split('-')
            
#             usage_val = val
#             # 🚀 [의도적 오류 시나리오] 정합성 엔진 테스트용
#             if site == 'Site B' and metric == 'lng_nm3' and date == '2025-01':
#                 usage_val = val / 1000.0  # ⚠️ 단위 오류 유도 (4번)
#             elif site == 'Site A' and metric == 'electricity_mwh' and date == '2025-03':
#                 usage_val = val + 150.0   # ⚠️ 수치 불일치 유도 (3번)
#             elif site == 'Site B' and metric == 'lng_nm3' and date == '2025-05':
#                 usage_val = val * 0.7     # ⚠️ 데이터 누락 유도 (3번)

#             # [해결포인트] 검증 함수가 바로 읽을 수 있는 '평면형 딕셔너리' 구조
#             preparsed_content = {
#                 "customer_number": str(cust_no),
#                 "year": year,
#                 "month": month,
#                 "usage": float(usage_val),
#                 "unit": 'MWh' if metric == 'electricity_mwh' else 'Nm3'
#             }
            
#             # 4개의 튜플 데이터 구성
#             mock_records.append((
#                 f"preparsed_inv_{cust_no}_{date}.json", # file_name
#                 json.dumps(preparsed_content, ensure_ascii=False), # raw_content
#                 'Preparsed_Mock_Engine', # ocr_provider
#                 'Pending' # processing_status
#             ))

#         # 3. DB 적재 (컬럼 4개와 VALUES 4개 일치 확인)
#         cursor.executemany("""
#             INSERT INTO raw_ocr_data (file_name, raw_content, ocr_provider, processing_status)
#             VALUES (?, ?, ?, ?)
#         """, mock_records)
        
#         conn.commit()
#     print(f"✅ 총 {len(mock_records)}건의 Pre-parsed 데이터가 성공적으로 재적재되었습니다.")

# if __name__ == "__main__":
#     inject_preparsed_mock_data_final()

✅ 총 45건의 Pre-parsed 데이터가 성공적으로 재적재되었습니다.
